In [138]:
from sklearn.model_selection import cross_val_score, KFold, cross_validate, RepeatedKFold
from sklearn.preprocessing import PowerTransformer, RobustScaler, StandardScaler
from sklearn.feature_selection import VarianceThreshold, RFECV, RFE
from sklearn.linear_model import LinearRegression, HuberRegressor


from ml_enhance import plot_scaled_linreg_result, CorrelationFilter


import matplotlib.pyplot as plt
from sklearn import pipeline
import seaborn as sns
import pandas as pd
import numpy as np
import scipy
import json

In [139]:
def make_pipeline(model, step_name):
    return pipeline.Pipeline([
        ("variance", VarianceThreshold(threshold=0.0)),
        ("remove_corr", CorrelationFilter(threshold=0.95)),
        ("transform", PowerTransformer(method='yeo-johnson', standardize=False)),
        ("scale", StandardScaler()),
        (step_name, model)
    ])

In [ ]:
pl_linear = make_pipeline(LinearRegression(), "predict")
pl_huber = make_pipeline(HuberRegressor(), "predict")

As the data (even after transformation and scaling) consists of a lot of outliers that cannot be removed (as they are valid molecules and are therefore part of the real world observables) I decided to use [Huber regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.HuberRegressor.html) instead of linear regression, this method is more robust to outliers.

In [141]:
df = pd.read_csv("../data/processed_dataset_wo_metals_w_even_more_qm.csv")

In [ ]:
y = df["solubility"]
X = df.drop(["solubility", "smiles", "canon_smiles", "id"], axis=1)

In [ ]:
X = X.drop([
        "avg_atomic_quadrupole_principal_invariant_3", # quadrupole principal invariant 3 features correlate highly with the invariant 2 features, so can drop them
        "max_atomic_quadrupole_principal_invariant_3",
        "molecular_quadrupole_principal_invariant_3",
        "avg_atomic_dipole_dipole_interaction" # the dipole dipole interaction between atoms would physically not be that influential on the solubility, can drop it
    ], axis=1)

In [144]:
with open("../data/rdkit_feature_names.json", "r") as f:
    rdkit_feature_list: list = json.load(f)

mask = X.columns.isin(rdkit_feature_list)

In [145]:
X_topo = X.iloc[:, mask]
X_qm = X.iloc[:, ~mask]

## Huber Regression: Topology VS. QM + Topology

In [167]:
inner_kf = KFold(n_splits=5, shuffle=True, random_state=42)
repeated_kf = RepeatedKFold(n_splits=5, n_repeats=5, random_state=15)

In [168]:
scoring = {
    "r2": "r2",
    "MSE": "neg_mean_squared_error"
}

In [169]:
param_grid = {
    "predict__epsilon": [1.1, 1.35, 1.5, 2.0],
    "predict__alpha": [1e-5, 1e-4, 1e-3, 1e-2]
}

In [171]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    estimator=pl_huber,
    param_grid=param_grid,
    cv=inner_kf,
    scoring='r2',
    n_jobs=4,
    verbose=11
)

In [ ]:
scores_combo = cross_validate(grid, X, y, cv=repeated_kf, scoring=scoring, n_jobs=4, return_estimator=True, return_train_score=True)

In [ ]:
scores_topo = cross_validate(grid, X_topo, y, cv=repeated_kf, scoring=scoring, n_jobs=4, return_estimator=True, return_train_score=True)

In [ ]:
scores_qm = cross_validate(grid, X_qm, y, cv=repeated_kf, scoring=scoring, n_jobs=4, return_estimator=True, return_train_score=True)

In [156]:
print(f"Train R2 scores:\nTopology alone: {scores_topo["train_r2"].mean()}\nQM alone: {scores_qm["train_r2"].mean()}\nCombined: {scores_combo["train_r2"].mean()}")
print("\n")
print(f"Test R2 scores:\nTopology alone: {scores_topo["test_r2"].mean()}\nQM alone: {scores_qm["test_r2"].mean()}\nCombined: {scores_combo["test_r2"].mean()}")

Train R2 scores:
Topology alone: 0.8262991760437502
QM alone: 0.775008823707611
Combined: 0.8440770885813674


Test R2 scores:
Topology alone: 0.8149067153443121
QM alone: 0.7620393936183701
Combined: 0.8239383800925194


In [164]:
from scipy.stats import ttest_rel

ttest_result = ttest_rel(scores_combo["test_r2"], scores_topo["test_r2"])

print("Topo mean R2:", scores_topo["test_r2"].mean())
print("Combined mean R2:", scores_combo["test_r2"].mean())
print("Mean improvement:", (scores_combo["test_r2"] - scores_topo["test_r2"]).mean())
print(f"p-value: {ttest_result[1]} ->{'not' if ttest_result[1] > 0.05 else ''} statistically significant")

Topo mean R2: 0.8149067153443121
Combined mean R2: 0.8239383800925194
Mean improvement: 0.009031664748207158
p-value: 0.0031675029535858596 -> statistically significant


In [159]:
print(f"Train MSE scores:\nTopology alone: {np.abs(scores_topo["train_MSE"]).mean()}\nQM alone: {np.abs(scores_qm["train_MSE"]).mean()}\nCombined: {np.abs(scores_combo["train_MSE"]).mean()}")
print("\n")
print(f"Test MSE scores:\nTopology alone: {np.abs(scores_topo["test_MSE"]).mean()}\nQM alone: {np.abs(scores_qm["test_MSE"]).mean()}\nCombined: {np.abs(scores_combo["test_MSE"]).mean()}")

Train MSE scores:
Topology alone: 0.9251251696815433
QM alone: 1.1982908401673646
Combined: 0.8304457321503875


Test MSE scores:
Topology alone: 0.9848945239449116
QM alone: 1.2657923782562783
Combined: 0.9369128569120904


In [163]:
ttest_result = ttest_rel(np.abs(scores_combo["test_MSE"]), np.abs(scores_topo["test_MSE"]))

print("Topo mean R2:", np.abs(scores_topo["test_MSE"]).mean())
print("Combined mean R2:", np.abs(scores_combo["test_MSE"]).mean())
print("Mean improvement:", (np.abs(scores_combo["test_MSE"]) - np.abs(scores_topo["test_MSE"])).mean())
print(f"p-value: {ttest_result[1]} ->{'not' if ttest_result[1] > 0.05 else ''} statistically significant")

Topo mean R2: 0.9848945239449116
Combined mean R2: 0.9369128569120904
Mean improvement: -0.04798166703282142
p-value: 0.0035364071333895833 -> statistically significant


Based on the results i got from the Huber regression, the QM descriptors alone seem to give the worst performance out of the three and **there seems to be a small significant difference between the topological descriptors alone and the combined feature set**. The combined set of features provides a slightly better prediction. Whether this is truely because of the QM descriptors, or because of model bias should be analyzed by looking at other models (RF, GAM, KRR?)

## Recursive Feature Elimination

Lets try and make a RFE to see whether the QM descriptors do in fact provide any value to the linear model.

In [35]:
def get_coef(estimator: LinearRegression):
    return estimator.named_steps['predict'].coef_

In [38]:
rfe.cv_results_

{'mean_test_score': array([ 3.42656517e-01,  4.58337156e-01,  5.74359498e-01,  5.94954891e-01,
        -1.24232774e+10, -2.09549346e+10, -1.78490282e+10, -1.60467847e+10,
        -1.37180294e+10, -1.34953940e+10, -1.60355167e+10, -1.65754138e+10,
        -1.67303027e+10, -1.68805609e+10, -1.54904673e+10, -1.37981350e+10,
        -1.34181780e+10, -1.42563546e+10, -1.45420551e+10, -1.50397853e+10,
        -1.52080845e+10, -1.40291478e+10, -1.23411224e+10, -1.14546996e+10,
        -1.12640548e+10, -1.15233881e+10, -1.22455757e+10, -1.34429544e+10,
        -1.28101685e+10, -1.28329902e+10, -1.18267381e+10, -1.10171808e+10,
        -1.12615381e+10, -1.11145175e+10, -8.43832819e+09, -8.61983526e+09,
        -8.40813157e+09, -8.14135669e+09, -8.95894984e+09, -8.99819640e+09,
        -8.87864057e+09, -8.85662967e+09, -8.85781160e+09, -8.59445870e+09,
        -8.52819275e+09, -8.62571231e+09, -8.74503805e+09, -8.24926735e+09,
        -8.31178459e+09, -8.26338364e+09, -8.17759630e+09, -8.6335880

In [39]:
filtered_features = rfe.get_feature_names_out()
X_rfe = X[filtered_features]
X_rfe.info()

<class 'pandas.DataFrame'>
RangeIndex: 8763 entries, 0 to 8762
Data columns (total 4 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   atomization_energy                           8763 non-null   float64
 1   avg_atomic_quadrupole_principal_invariant_2  8763 non-null   float64
 2   min_atomic_quadrupole_principal_invariant_2  8763 non-null   float64
 3   std_atomic_quadrupole_principal_invariant_2  8763 non-null   float64
dtypes: float64(4)
memory usage: 274.0 KB


In [40]:
inner_kf = KFold(n_splits=5, shuffle=True, random_state=40)
rfe_pl = make_pipeline(RFECV(LinearRegression(), cv=inner_kf, scoring="r2", n_jobs=4), 'rfe')
rfe_pl.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('variance', ...), ('remove_corr', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"threshold threshold: float, default=0Features with a training-set variance lower than this threshold willbe removed. The default is to keep all features with non-zero variance,i.e. remove the features that have the same value in all samples.",0.0
,threshold,0.95
,"method method: {'yeo-johnson', 'box-cox'}, default='yeo-johnson'The power transform method. Available methods are:- 'yeo-johnson' [1]_, works with positive and negative values- 'box-cox' [2]_, only works with strictly positive values",'yeo-johnson'
,"standardize standardize: bool, default=TrueSet to True to apply zero-mean, unit-variance normalization to thetransformed output.",False
,"copy copy: bool, default=TrueSet to False to perform inplace computation during transformation.",True
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True


In these remaining features, I am curious how many QM features are still left. 

In [ ]:
with open("../data/properties.json", "r") as f:
    data = json.load(f)

qm_feature_names = [feature["name_of_property"] for feature in data]

In [ ]:
scores_pre_rfe = cross_validate(pl_linear, X, y, cv=cv, scoring=scoring, n_jobs=4, return_estimator=True, return_train_score=True)

In [ ]:
scores_post_rfe = cross_validate(pl_linear, X_rfe, y, cv=cv, scoring=scoring, n_jobs=4, return_estimator=True, return_train_score=True)

In [ ]:
train_score_pre_rfe = scores_pre_rfe["train_r2"].mean()
test_score_pre_rfe = scores_pre_rfe["test_r2"].mean()
print(f"Pre-RFE:\nTrain score : {train_score_pre_rfe}\nTest score: {test_score_pre_rfe}\nDifference: {abs(test_score_pre_rfe - train_score_pre_rfe)}")

print("\n")

train_score_rfe = scores_post_rfe['train_r2'].mean()
test_score_rfe = scores_post_rfe['test_r2'].mean()
print(f"Post-RFE:\nTrain score : {train_score_rfe}\nTest score: {test_score_rfe}\nDifference: {abs(test_score_rfe - train_score_rfe)}")

Pre-RFE:
Train score : 0.8486718410870815
Test score: 0.8215690359567269
Difference: 0.027102805130354568


Post-RFE:
Train score : 0.848383203926588
Test score: 0.8229526949889514
Difference: 0.025430508937636542


The difference between the test and train score is for both situations small => no overfitting

In [ ]:
print(f"Pre-RFE R2 score: {scores_pre_rfe['test_r2'].mean()}\nPost-RFE R2 score: {scores_post_rfe['test_r2'].mean()}\nMean improvement: {(scores_post_rfe['test_r2'] - scores_pre_rfe['test_r2']).mean()}")
p_value = ttest_rel(scores_pre_rfe['test_r2'], scores_post_rfe['test_r2'])[1]
print(f"p-value: {p_value}")

Pre-RFE R2 score: 0.8215690359567269
Post-RFE R2 score: 0.8229526949889514
Mean improvement: 0.0013836590322244603
p-value: 0.12216066972413299


Despite the fact that the mean improvement of $R^2$ is so small, the increase is statistically significant. By reducing the amount of features with the RFE, we got an increase in performance.

In [ ]:
def get_best_pl(fit):
    return fit["estimator"][fit["test_r2"].argmax()]    

In [ ]:
best_post_rfe_pl = get_best_pl(scores_post_rfe)
best_pre_rfe_pl = get_best_pl(scores_pre_rfe)